# Phase 11 — Testing PySpark Experiments Notebook

This is the **worked SOLUTION notebook** for Phase 11.

Run it **top-to-bottom**. The repeated workflow is:

```text
state the contract
    ↓
choose the smallest deterministic fixture
    ↓
classify the test boundary
    ↓
predict the correct output
    ↓
call production logic
    ↓
assert schema / identity / grain / values
    ↓
reconcile where applicable
    ↓
introduce edge cases
    ↓
prove a regression is detected
```

Core question:

> **Can the test suite detect a meaningful pipeline defect before downstream data is silently corrupted?**

Important:

- Examples target **PySpark 4.2.0**.
- Python strings are single-quoted.
- Important code includes inline teaching comments.
- Existing Phase 9/10 logic is treated as the subject under test.
- Small deterministic fixtures are preferred.
- DataFrame row order is never assumed implicitly.
- The notebook does **not** perform the formal Phase 11 mastery gate, update `ROADMAP.md`, mark Phase 11 complete, or enter Phase 12.


<a id="toc"></a>
## Table of Contents

- [Setup and Existing Subjects Under Test](#setup)
- [Testing Exercise Protocol](#protocol)
- [Experiment 1 — Reusable Spark Fixture](#experiment-1)
- [Experiment 2 — Unit Test a Filter Transformation](#experiment-2)
- [Experiment 3 — Deterministic DataFrame Comparison](#experiment-3)
- [Experiment 4 — Schema Testing](#experiment-4)
- [Experiment 5 — Row-Count vs. Identity Assertions](#experiment-5)
- [Experiment 6 — Grain and Primary-Key Testing](#experiment-6)
- [Experiment 7 — Composite-Key Testing](#experiment-7)
- [Experiment 8 — Referential-Integrity Testing](#experiment-8)
- [Experiment 9 — Numerical Reconciliation](#experiment-9)
- [Experiment 10 — Accepted/Rejected Quality Testing](#experiment-10)
- [Experiment 11 — Deterministic Window Testing](#experiment-11)
- [Experiment 12 — Edge Cases](#experiment-12)
- [Experiment 13 — Integration Testing](#experiment-13)
- [Experiment 14 — Distributed Spark Testing Cost](#experiment-14)
- [Experiment 15 — Regression Detection](#experiment-15)
- [Applied Phase 11 Project](#applied-project)
- [Cleanup](#cleanup)


<a id="setup"></a>
# Setup and Existing Subjects Under Test

Phase 11 protects logic already created in Phases 9 and 10.

```text
Phase 9
→ modular transformation architecture

Phase 10
→ explicit validation contracts

Phase 11
→ automated regression protection
```

[Back to Table of Contents](#toc)


In [ ]:
from datetime import date
from decimal import Decimal
import importlib.util
from pathlib import Path
from types import ModuleType

from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType
from pyspark.sql.types import StringType
from pyspark.sql.types import StructField
from pyspark.sql.types import StructType
from pyspark.testing.utils import assertDataFrameEqual
from pyspark.testing.utils import assertSchemaEqual


In [ ]:
spark = (
    SparkSession.builder
    .appName('phase_11_testing_pyspark_notebook')
    .master('local[2]')
    # Keep tiny test shuffles predictable and inexpensive.
    .config('spark.sql.shuffle.partitions', '2')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')


In [ ]:
def load_module_from_path(
    module_name: str,
    module_path: Path,
) -> ModuleType:
    '''Load one existing phase lecture module from its repository path.'''

    # WHAT: load existing preserved Phase 9/10 logic.
    # WHY: tests should call real production/reusable logic rather than copies.
    spec = importlib.util.spec_from_file_location(
        module_name,
        module_path,
    )

    if spec is None or spec.loader is None:
        raise ImportError(f'Unable to load module from {module_path}.')

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    return module


In [ ]:
# Resolve the repository root from the current working directory.
repo_root = Path.cwd()

if repo_root.name == 'exercises':
    repo_root = repo_root.parents[2]
elif repo_root.name == 'phase_11_testing_pyspark':
    repo_root = repo_root.parents[1]

phase_09 = load_module_from_path(
    'phase_09_lecture_for_testing',
    repo_root
    / 'phases'
    / 'phase_09_pyspark_application_architecture'
    / 'phase_09_lecture.py',
)

phase_10 = load_module_from_path(
    'phase_10_lecture_for_testing',
    repo_root
    / 'phases'
    / 'phase_10_data_quality_schema_enforcement'
    / 'phase_10_lecture.py',
)


In [ ]:
def assert_unique_key(
    df: DataFrame,
    key_columns: list[str],
) -> None:
    '''Assert that the declared key identifies at most one row.'''

    # groupBy() is intentional because uniqueness is a dataset-level property.
    duplicate_exists = (
        df
        .groupBy(*key_columns)
        .count()
        .filter(F.col('count') > 1)
        .limit(1)
        .count()
        > 0
    )

    assert not duplicate_exists


def assert_no_orphans(
    child_df: DataFrame,
    parent_df: DataFrame,
    key_columns: list[str],
) -> None:
    '''Assert that every child key matches a parent key.'''

    # left_anti isolates child keys with no parent match.
    orphan_exists = (
        child_df
        .select(*key_columns)
        .join(
            parent_df.select(*key_columns),
            on=key_columns,
            how='left_anti',
        )
        .limit(1)
        .count()
        > 0
    )

    assert not orphan_exists


def assert_measure_reconciles(
    input_df: DataFrame,
    output_df: DataFrame,
    column_name: str,
) -> None:
    '''Assert exact measure conservation across a transformation boundary.'''

    input_value = (
        input_df
        .agg(F.sum(column_name).alias('value'))
        .first()['value']
    )

    output_value = (
        output_df
        .agg(F.sum(column_name).alias('value'))
        .first()['value']
    )

    assert input_value == output_value


<a id="protocol"></a>
# Testing Exercise Protocol

For every experiment:

```text
1. State the contract.
2. Identify unit vs. integration scope.
3. State input and output grain.
4. Use the smallest deterministic fixture.
5. Predict the result before execution.
6. Call real production logic.
7. Assert business/data correctness.
8. Explain Spark actions triggered by the assertion.
9. Identify the defect the test protects against.
```

[Back to Table of Contents](#toc)


<a id="experiment-1"></a>
# Experiment 1 — Reusable Spark Fixture

A real `pytest` suite should centralize the shared Spark runtime in `conftest.py`.

The notebook already created one shared Spark session above, so this experiment shows the exact fixture pattern you would preserve in the actual test suite.

[Back to Table of Contents](#toc)


In [ ]:
PYTEST_SPARK_FIXTURE = '''
import pytest

from pyspark.sql import SparkSession


@pytest.fixture(scope='session')
def spark():
    # Reuse one Spark runtime across the complete pytest session.
    spark_session = (
        SparkSession.builder
        .master('local[2]')
        .appName('phase_11_tests')
        .config('spark.sql.shuffle.partitions', '2')
        .getOrCreate()
    )

    spark_session.sparkContext.setLogLevel('WARN')

    yield spark_session

    # Clean up the shared runtime after all tests finish.
    spark_session.stop()
'''

print(PYTEST_SPARK_FIXTURE)


### Key point

```text
SparkSession
→ expensive shared test runtime

scenario DataFrame
→ small focused fixture
```

Use session scope for the Spark runtime, but keep scenario-specific mutable data isolated.

[Back to Table of Contents](#toc)


<a id="experiment-2"></a>
# Experiment 2 — Unit Test a Filter Transformation

Contract:

```text
filter_orders()
→ order_status must be included
→ net_sales must meet the configured minimum
→ surviving order grain remains one row per order_id
```

[Back to Table of Contents](#toc)


In [ ]:
filter_schema = StructType(
    [
        StructField('order_id', StringType(), nullable=False),
        StructField('customer_id', StringType(), nullable=False),
        StructField('order_status', StringType(), nullable=False),
        StructField('net_sales', DecimalType(12, 2), nullable=False),
    ]
)

filter_orders_df = spark.createDataFrame(
    [
        # Keep.
        ('O001', 'C001', 'COMPLETED', Decimal('125.00')),
        # Exclude by status.
        ('O002', 'C002', 'CANCELLED', Decimal('80.00')),
        # Exclude by amount.
        ('O003', 'C003', 'COMPLETED', Decimal('-1.00')),
    ],
    schema=filter_schema,
)

actual_filtered_df = phase_09.filter_orders(
    filter_orders_df,
    included_statuses=('COMPLETED',),
    minimum_net_sales=Decimal('0.00'),
)

actual_order_ids = {
    row['order_id']
    for row in actual_filtered_df.select('order_id').collect()
}

assert actual_order_ids == {'O001'}
assert_unique_key(actual_filtered_df, ['order_id'])

actual_filtered_df.show(truncate=False)


### Why this is a unit test

Only one production responsibility is exercised:

```text
filter_orders()
```

No reader, writer, validation layer, or aggregation is required.

[Back to Table of Contents](#toc)


<a id="experiment-3"></a>
# Experiment 3 — Deterministic DataFrame Comparison

Spark DataFrames are logically unordered unless ordering is explicitly requested.

[Back to Table of Contents](#toc)


In [ ]:
comparison_schema = StructType(
    [
        StructField('order_id', StringType(), nullable=False),
        StructField('status', StringType(), nullable=False),
    ]
)

actual_df = spark.createDataFrame(
    [
        ('O002', 'COMPLETED'),
        ('O001', 'COMPLETED'),
    ],
    schema=comparison_schema,
)

expected_df = spark.createDataFrame(
    [
        ('O001', 'COMPLETED'),
        ('O002', 'COMPLETED'),
    ],
    schema=comparison_schema,
)

# Strategy 1: deterministically sort tiny assertion data.
assert (
    actual_df.orderBy('order_id').collect()
    == expected_df.orderBy('order_id').collect()
)

# Strategy 2: compare DataFrames logically without requiring row order.
assertDataFrameEqual(
    actual_df,
    expected_df,
    checkRowOrder=False,
)

print('Deterministic comparison passed.')


### Do not do this in production merely for testing

```python
production_df.orderBy(...)
```

Global `orderBy()` can require a shuffle.

Ordering is added here only to make a **tiny test assertion** deterministic.

[Back to Table of Contents](#toc)


<a id="experiment-4"></a>
# Experiment 4 — Schema Testing

Test structure and values separately.

[Back to Table of Contents](#toc)


In [ ]:
source_df = spark.createDataFrame(
    [
        ('ON', Decimal('200.00')),
    ],
    schema=StructType(
        [
            StructField('province', StringType(), nullable=False),
            StructField('net_sales', DecimalType(12, 2), nullable=False),
        ]
    ),
)

actual_df = phase_09.add_processing_date(
    source_df,
    run_date=date(2026, 9, 7),
)

actual_types = {
    field.name: field.dataType.simpleString()
    for field in actual_df.schema.fields
}

# Structural assertion.
assert actual_types['processing_date'] == 'date'

# Data assertion.
assert actual_df.first()['processing_date'] == date(2026, 9, 7)

actual_df.printSchema()
actual_df.show(truncate=False)


In [ ]:
expected_schema = actual_df.schema

# Demonstrate the dedicated schema assertion helper.
assertSchemaEqual(actual_df.schema, expected_schema)

print('Schema comparison passed.')


<a id="experiment-5"></a>
# Experiment 5 — Row-Count vs. Identity Assertions

A correct count can coexist with incorrect business identity.

[Back to Table of Contents](#toc)


In [ ]:
count_schema = StructType(
    [
        StructField('order_id', StringType(), nullable=False),
    ]
)

correct_df = spark.createDataFrame(
    [('O001',), ('O002',)],
    schema=count_schema,
)

wrong_df = spark.createDataFrame(
    [('O003',), ('O004',)],
    schema=count_schema,
)

# Both pass the same count assertion.
assert correct_df.count() == 2
assert wrong_df.count() == 2

# Business identity distinguishes them.
correct_ids = {
    row['order_id']
    for row in correct_df.collect()
}
wrong_ids = {
    row['order_id']
    for row in wrong_df.collect()
}

assert correct_ids == {'O001', 'O002'}
assert wrong_ids != correct_ids

print('Count alone was insufficient; identity assertion exposed the difference.')


<a id="experiment-6"></a>
# Experiment 6 — Grain and Primary-Key Testing

Declared grain:

```text
orders_df
= one row per order_id
```

[Back to Table of Contents](#toc)


In [ ]:
unique_orders_df = spark.createDataFrame(
    [
        ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('10.00')),
        ('O002', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('20.00')),
    ],
    schema=phase_10.ORDERS_SCHEMA,
)

assert_unique_key(unique_orders_df, ['order_id'])
print('Unique order grain passed.')


In [ ]:
duplicate_orders_df = spark.createDataFrame(
    [
        ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('10.00')),
        ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('15.00')),
    ],
    schema=phase_10.ORDERS_SCHEMA,
)

duplicate_keys_df = phase_10.find_duplicate_keys(
    duplicate_orders_df,
    key_columns=['order_id'],
)

assert duplicate_keys_df.count() == 1
assert duplicate_keys_df.first()['order_id'] == 'O001'

duplicate_keys_df.show(truncate=False)


### Important distinction

```text
exact duplicate row
!=
duplicate business key
```

Two `O001` rows can violate order grain even when their other values differ.

[Back to Table of Contents](#toc)


<a id="experiment-7"></a>
# Experiment 7 — Composite-Key Testing

Inventory grain:

```text
snapshot_date + store_id + product_id
```

[Back to Table of Contents](#toc)


In [ ]:
inventory_df = spark.createDataFrame(
    [
        (date(2026, 9, 5), 'S001', 'P001', 10),
        (date(2026, 9, 5), 'S001', 'P001', 12),
        (date(2026, 9, 5), 'S001', 'P002', 7),
    ],
    schema=phase_10.INVENTORY_SCHEMA,
)

validated_inventory_df = phase_10.add_inventory_reasons(inventory_df)

duplicate_inventory_rows = (
    validated_inventory_df
    .filter(
        (F.col('store_id') == 'S001')
        & (F.col('product_id') == 'P001')
    )
    .collect()
)

assert len(duplicate_inventory_rows) == 2

for row in duplicate_inventory_rows:
    assert 'DUPLICATE_INVENTORY_KEY' in row['rejection_reasons']

validated_inventory_df.show(truncate=False)


### The key is the combination

It is completely normal for:

```text
snapshot_date
store_id
product_id
```

to repeat individually.

The combination must be unique.

[Back to Table of Contents](#toc)


<a id="experiment-8"></a>
# Experiment 8 — Referential-Integrity Testing

Relationship:

```text
orders.customer_id
    →
customers.customer_id
```

[Back to Table of Contents](#toc)


In [ ]:
orders_df = spark.createDataFrame(
    [
        ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('25.00')),
        ('O002', 'C999', date(2026, 9, 1), 'COMPLETED', Decimal('30.00')),
    ],
    schema=phase_10.ORDERS_SCHEMA,
)

customers_df = spark.createDataFrame(
    [
        ('C001', 'Alice Wong', 'ON', 'CONSUMER'),
    ],
    schema=phase_10.CUSTOMERS_SCHEMA,
)

validated_df = phase_10.validate_orders(
    orders_df,
    customers_df,
)

accepted_df, rejected_df = phase_10.split_accepted_rejected(
    validated_df
)

accepted_ids = {
    row['order_id']
    for row in accepted_df.select('order_id').collect()
}

assert accepted_ids == {'O001'}

orphan_row = (
    rejected_df
    .filter(F.col('order_id') == 'O002')
    .first()
)

assert 'ORPHAN_CUSTOMER_ID' in orphan_row['rejection_reasons']

# Accepted child rows must have parents.
assert_no_orphans(
    accepted_df,
    customers_df,
    ['customer_id'],
)

validated_df.show(truncate=False)


### Separate contracts

```text
referential integrity
→ parent exists

parent grain
→ expected parent key is unique
```

Both matter before a downstream join can safely assume one parent per key.

[Back to Table of Contents](#toc)


<a id="experiment-9"></a>
# Experiment 9 — Numerical Reconciliation

Row counts can remain correct while measures become wrong.

[Back to Table of Contents](#toc)


In [ ]:
sales_input_df = spark.createDataFrame(
    [
        ('O001', 'ON', Decimal('125.00')),
        ('O002', 'ON', Decimal('75.00')),
        ('O003', 'BC', Decimal('200.00')),
    ],
    schema=StructType(
        [
            StructField('order_id', StringType(), nullable=False),
            StructField('province', StringType(), nullable=False),
            StructField('net_sales', DecimalType(12, 2), nullable=False),
        ]
    ),
)

sales_by_province_df = phase_09.build_sales_by_province(
    sales_input_df
)

assert_measure_reconciles(
    sales_input_df,
    sales_by_province_df,
    column_name='net_sales',
)

expected_df = spark.createDataFrame(
    [
        ('BC', 1, Decimal('200.00')),
        ('ON', 2, Decimal('200.00')),
    ],
    ['province', 'order_count', 'net_sales'],
)

actual_rows = [
    (row['province'], row['order_count'], row['net_sales'])
    for row in sales_by_province_df.orderBy('province').collect()
]

expected_rows = [
    ('BC', 1, Decimal('200.00')),
    ('ON', 2, Decimal('200.00')),
]

assert actual_rows == expected_rows

sales_by_province_df.show(truncate=False)


### Per-row reconciliation pattern

For a sales fact:

```text
gross_margin
=
gross_sales - gross_cost
```

A professional test can assert that no accepted row violates the formula.

[Back to Table of Contents](#toc)


In [ ]:
fact_sales_df = spark.createDataFrame(
    [
        ('S001', Decimal('100.00'), Decimal('70.00'), Decimal('30.00')),
        ('S002', Decimal('50.00'), Decimal('20.00'), Decimal('30.00')),
    ],
    ['sale_id', 'gross_sales', 'gross_cost', 'gross_margin'],
)

invalid_margin_df = fact_sales_df.filter(
    F.col('gross_margin')
    != (F.col('gross_sales') - F.col('gross_cost'))
)

assert invalid_margin_df.count() == 0

print('Gross-margin reconciliation passed.')


<a id="experiment-10"></a>
# Experiment 10 — Accepted/Rejected Quality Testing

Tests should protect both sides of the Phase 10 split.

[Back to Table of Contents](#toc)


In [ ]:
orders_df = spark.createDataFrame(
    [
        # Valid.
        ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('25.00')),
        # Multiple row-level defects.
        ('O002', None, date(2026, 9, 1), 'COMPLETED', Decimal('-1.00')),
        # Orphan.
        ('O003', 'C999', date(2026, 9, 1), 'COMPLETED', Decimal('10.00')),
    ],
    schema=phase_10.ORDERS_SCHEMA,
)

customers_df = spark.createDataFrame(
    [
        ('C001', 'Alice Wong', 'ON', 'CONSUMER'),
    ],
    schema=phase_10.CUSTOMERS_SCHEMA,
)

validated_df = phase_10.validate_orders(
    orders_df,
    customers_df,
)

accepted_df, rejected_df = phase_10.split_accepted_rejected(
    validated_df
)

assert orders_df.count() == accepted_df.count() + rejected_df.count()

assert {
    row['order_id']
    for row in accepted_df.select('order_id').collect()
} == {'O001'}

multi_reason_row = (
    rejected_df
    .filter(F.col('order_id') == 'O002')
    .first()
)

assert set(multi_reason_row['rejection_reasons']) == {
    'MISSING_CUSTOMER_ID',
    'NEGATIVE_NET_SALES',
}

orphan_row = (
    rejected_df
    .filter(F.col('order_id') == 'O003')
    .first()
)

assert 'ORPHAN_CUSTOMER_ID' in orphan_row['rejection_reasons']

validated_df.show(truncate=False)


<a id="experiment-11"></a>
# Experiment 11 — Deterministic Window Testing

A deterministic test must include the tie it claims to protect.

[Back to Table of Contents](#toc)


In [ ]:
history_df = spark.createDataFrame(
    [
        ('R001', 'C001', date(2026, 5, 1), 'ON'),
        ('R002', 'C001', date(2026, 5, 1), 'QC'),
    ],
    schema=phase_09.CUSTOMER_HISTORY_SCHEMA,
)

latest_df = phase_09.select_latest_customer_record(history_df)

latest_row = latest_df.first()

# Same effective date, so customer_record_id DESC decides the winner.
assert latest_row['customer_record_id'] == 'R002'
assert latest_row['province'] == 'QC'

latest_df.show(truncate=False)


### Why this fixture matters

If the two effective dates were different, the test would prove only:

```text
latest date wins
```

It would **not** prove:

```text
ties are resolved deterministically
```

[Back to Table of Contents](#toc)


<a id="experiment-12"></a>
# Experiment 12 — Edge Cases

Test boundaries deliberately.

[Back to Table of Contents](#toc)


In [ ]:
# Empty input requires an explicit schema.
empty_orders_df = spark.createDataFrame(
    [],
    schema=phase_10.ORDERS_SCHEMA,
)

empty_customers_df = spark.createDataFrame(
    [],
    schema=phase_10.CUSTOMERS_SCHEMA,
)

validated_empty_df = phase_10.validate_orders(
    empty_orders_df,
    empty_customers_df,
)

accepted_empty_df, rejected_empty_df = (
    phase_10.split_accepted_rejected(validated_empty_df)
)

assert validated_empty_df.count() == 0
assert accepted_empty_df.count() == 0
assert rejected_empty_df.count() == 0

print('Empty-input behavior passed.')


In [ ]:
boundary_df = spark.createDataFrame(
    [
        ('O001', 'C001', 'COMPLETED', Decimal('-0.01')),
        ('O002', 'C001', 'COMPLETED', Decimal('0.00')),
        ('O003', 'C001', 'COMPLETED', Decimal('0.01')),
    ],
    schema=filter_schema,
)

actual_boundary_df = phase_09.filter_orders(
    boundary_df,
    included_statuses=('COMPLETED',),
    minimum_net_sales=Decimal('0.00'),
)

assert {
    row['order_id']
    for row in actual_boundary_df.select('order_id').collect()
} == {'O002', 'O003'}

actual_boundary_df.show(truncate=False)


<a id="experiment-13"></a>
# Experiment 13 — Integration Testing

Integration means multiple meaningful responsibilities work together.

It does **not** mean large-scale data.

[Back to Table of Contents](#toc)


In [ ]:
orders_df = spark.createDataFrame(
    [
        ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('100.00')),
        ('O002', 'C002', date(2026, 9, 1), 'COMPLETED', Decimal('50.00')),
        ('O003', 'C999', date(2026, 9, 1), 'COMPLETED', Decimal('25.00')),
    ],
    schema=phase_10.ORDERS_SCHEMA,
)

customers_df = spark.createDataFrame(
    [
        ('C001', 'Alice Wong', 'ON', 'CONSUMER'),
        ('C002', 'Carla Singh', 'BC', 'BUSINESS'),
    ],
    schema=phase_10.CUSTOMERS_SCHEMA,
)

# 1. Validate.
validated_df = phase_10.validate_orders(
    orders_df,
    customers_df,
)

# 2. Classify.
accepted_df, rejected_df = phase_10.split_accepted_rejected(
    validated_df
)

# 3. Transform accepted data.
enriched_df = phase_10.transform_accepted_orders(
    accepted_df,
    customers_df,
)

# 4. Aggregate.
result_df = phase_09.build_sales_by_province(
    enriched_df
)

actual_rows = [
    (row['province'], row['order_count'], row['net_sales'])
    for row in result_df.orderBy('province').collect()
]

assert actual_rows == [
    ('BC', 1, Decimal('50.00')),
    ('ON', 1, Decimal('100.00')),
]

# Classification reconciliation.
assert orders_df.count() == accepted_df.count() + rejected_df.count()

# Measure reconciliation.
assert_measure_reconciles(
    accepted_df,
    result_df,
    column_name='net_sales',
)

# Output grain.
assert_unique_key(result_df, ['province'])

result_df.show(truncate=False)


### Integration boundary proved

```text
Phase 10 validation
→ accepted/rejected split
→ enrichment
→ Phase 9 aggregation
→ reconciliation
```

[Back to Table of Contents](#toc)


<a id="experiment-14"></a>
# Experiment 14 — Distributed Spark Testing Cost

Assertions can still trigger Spark jobs.

[Back to Table of Contents](#toc)


In [ ]:
action_demo_df = spark.range(5)

# Each of these is an action.
row_count = action_demo_df.count()
first_row = action_demo_df.first()
all_rows = action_demo_df.collect()

assert row_count == 5
assert first_row['id'] == 0
assert len(all_rows) == 5

print('count() -> action')
print('first() -> action')
print('collect() -> action')


### Testing rule

```text
tiny deterministic fixture + collect()
→ reasonable

large production population + collect()
→ unsafe
```

The test suite should use tiny data rather than normalize unsafe driver collection patterns.

[Back to Table of Contents](#toc)


<a id="experiment-15"></a>
# Experiment 15 — Regression Detection

This demonstrates the Phase 11 mastery idea:

```text
passing contract
→ deliberate bad change
→ automated assertion fails
```

[Back to Table of Contents](#toc)


In [ ]:
def deliberately_broken_sales_by_province(
    enriched_orders_df: DataFrame,
) -> DataFrame:
    '''Deliberately wrong aggregation used only for regression testing.'''

    # WRONG ON PURPOSE:
    # The production contract requires SUM, not AVG.
    return (
        enriched_orders_df
        .groupBy('province')
        .agg(
            F.countDistinct('order_id').alias('order_count'),
            F.avg('net_sales').alias('net_sales'),
        )
    )


regression_input_df = spark.createDataFrame(
    [
        ('O001', 'ON', Decimal('125.00')),
        ('O002', 'ON', Decimal('75.00')),
        ('O003', 'BC', Decimal('200.00')),
    ],
    schema=StructType(
        [
            StructField('order_id', StringType(), nullable=False),
            StructField('province', StringType(), nullable=False),
            StructField('net_sales', DecimalType(12, 2), nullable=False),
        ]
    ),
)

broken_df = deliberately_broken_sales_by_province(
    regression_input_df
)

regression_detected = False

try:
    assert_measure_reconciles(
        regression_input_df,
        broken_df,
        column_name='net_sales',
    )
except AssertionError:
    regression_detected = True

assert regression_detected

print('PASS | deliberate SUM -> AVG corruption was detected.')


### Why this matters

A row-count-only test might still pass because the broken aggregation can produce the same number of province rows.

The reconciliation assertion protects the **measure**, not merely the shape.

[Back to Table of Contents](#toc)


<a id="applied-project"></a>
# Applied Phase 11 Project

Now protect the existing retail pipeline with a real `pytest` suite.

Suggested structure:

```text
tests/
├── conftest.py
├── test_transformations.py
├── test_validation.py
└── test_pipeline_integration.py
```

The suite should protect:

```text
schema contracts
transformation correctness
deterministic output
row population
grain
primary/composite-key uniqueness
referential integrity
accepted/rejected behavior
numerical reconciliation
edge cases
integration between meaningful components
```

[Back to Table of Contents](#toc)


## Applied Task Requirements

### `conftest.py`

Create:

```text
session-scoped Spark fixture
local[2]
small shuffle partition count
clean shutdown
```

### `test_transformations.py`

Protect at minimum:

```text
filter_orders()
build_sales_by_province()
add_processing_date()
select_latest_customer_record()
```

### `test_validation.py`

Protect at minimum:

```text
required-field rejection
domain/range rejection
duplicate order_id
duplicate composite inventory key
orphan customer_id
multiple rejection reasons
accepted/rejected reconciliation
```

### `test_pipeline_integration.py`

Protect:

```text
validation
→ accepted/rejected split
→ enrichment
→ aggregation
→ schema / grain / values / totals
```

### Regression requirement

Start with the suite passing.

Deliberately break one meaningful contract, for example:

```text
sum(net_sales)
→
avg(net_sales)
```

Confirm the relevant test fails.

Restore the correct implementation.

Confirm the suite passes again.

This is still practice; formal mastery is reviewed separately.

[Back to Table of Contents](#toc)


<a id="cleanup"></a>
# Cleanup

[Back to Table of Contents](#toc)


In [ ]:
spark.stop()
print('Spark session stopped.')
